# 04 · Clientes e Vulto+ da Vulto: raio-x

Fontes: `clientes_clientes.csv`, `clientes_antigos.csv` e `clientes_resgates.csv` (abas de `clientes_fidelidade_FINAL_v3 (1).xlsx`), `banco_pontos_mov.csv` e `banco_compras.csv` (do `vulto.db`).

Objetivo: **entender** os clientes e os pontos, sem decidir nada. As mesmas 4 perguntas.

**Regras do programa** (aba "LEIA ISSO" da planilha, jun/2025): R$ 1 = 1 ponto na loja física; site e app valem **dobro**; Ouro ganha **+50%**; **100 pontos = R$ 5** (era R$ 10 até fev/2025); pontos **vencem em 12 meses** (18 para Ouro, "ainda não tá no sistema"); resgate mínimo de 100 pontos; não pode ficar negativo ("o sistema deixa"); pontos só com CPF. **A coluna `Pontos` da aba Clientes foi congelada em 01/01/2025**: depois disso os pontos ficam no banco (`pontos_mov`). Algumas lojas continuaram anotando resgates na planilha. O banco ainda usa R$ 0,10 por ponto (`config.valor_ponto`).

> Antes de commitar: *Kernel → Restart & Clear Output* (as tabelas têm CPF, e-mail e telefone).

In [ ]:
from pathlib import Path

import pandas as pd

# raiz do repositório, de onde quer que o notebook tenha sido aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "migracao_vulto").is_dir())
STAGING = RAIZ / "staging_vulto"

# mostra todas as colunas nas tabelas
pd.set_option("display.max_columns", None)

In [ ]:
clientes = pd.read_csv(STAGING / "clientes_clientes.csv", dtype=str)
antigos = pd.read_csv(STAGING / "clientes_antigos.csv", dtype=str)
resgates = pd.read_csv(STAGING / "clientes_resgates.csv", dtype=str)
pontos = pd.read_csv(STAGING / "banco_pontos_mov.csv", dtype=str)
compras = pd.read_csv(STAGING / "banco_compras.csv", dtype=str)
pontos["q"], compras["v"] = pd.to_numeric(pontos["qtd"]), pd.to_numeric(compras["valor"])

## 1. O que é cada linha?

- `clientes`: um cadastro do Vulto+ (não uma pessoa: a mesma pessoa pode ter vários cadastros).
- `antigos`: um cliente do programa de 2023.
- `resgates`: um resgate anotado na planilha pela loja.
- `pontos`: um movimento de pontos no banco: C crédito, D resgate, E estorno, AJ ajuste.
- `compras`: uma compra que pode gerar pontos (o "lado fidelidade" da venda).

In [ ]:
for nome, tabela in [("clientes", clientes), ("antigos", antigos), ("resgates", resgates), ("pontos", pontos), ("compras", compras)]:
    print(nome, tabela.shape)
# a mesma pessoa em mais de um cadastro
print("CPFs com 2 ou mais cadastros:", clientes["cpf"].dropna().value_counts().gt(1).sum(),
      "| e-mails repetidos:", clientes["E-mail"].dropna().str.lower().duplicated().sum())
clientes[clientes["cpf"].isin(clientes["cpf"].dropna().value_counts().head(2).index)].sort_values("cpf")

## 2. O que significa cada coluna?

In [ ]:
# Pontos da planilha: vários formatos (9 = qualquer dígito)
display(clientes["Pontos"].dropna().str.replace(r"\d", "9", regex=True).value_counts().head(8))
for coluna in ["categoria", "aceita_msg", "loja_cadastro"]:
    print(coluna, "|", clientes[coluna].value_counts(dropna=False).to_dict())
cadastro = pd.to_datetime(clientes["Data Cadastro"], errors="coerce")
print("cadastrados a partir de 2025:", (cadastro >= "2025-01-01").sum(),
      "| desses com Pontos na planilha (que foi congelada em 01/01/2025):", ((cadastro >= "2025-01-01") & clientes["Pontos"].notna() & (clientes["Pontos"] != "0")).sum())

In [ ]:
# pontos no banco por tipo, e o período
print("período:", pontos["criado_em"].min(), "a", pontos["criado_em"].max())
pontos.groupby("tipo")["q"].agg(["count", "sum"])

## 3. Que problemas aparecem?

In [ ]:
# pontos creditados / valor da compra: deveria ser 1 na loja física e 2 no site e no app
creditos = pontos[pontos["tipo"] == "C"].assign(compra=lambda d: d["compra_id"].str.removesuffix(".0")).merge(
    compras[["id", "v", "origem", "canal"]], left_on="compra", right_on="id", suffixes=("", "_compra"))
creditos["pontos_por_real"] = (creditos["q"] / creditos["v"]).round(2)
# no app o valor está em centavos: 0,02 = 2 pontos por real; 0,01 = 1 ponto por real
pd.crosstab(creditos["canal"].str.lower(), creditos["pontos_por_real"])

In [ ]:
# bug do app: pontos em dobro sobre o valor em CENTAVOS (pontos_por_real = 2 com valor acima de 5 mil)
inflados = creditos[(creditos["origem"] == "app") & (creditos["pontos_por_real"] == 2) & (creditos["v"] > 5000)]
ajustes_bug = pontos[(pontos["tipo"] == "AJ") & pontos["obs"].fillna("").str.contains("bug")]
print("créditos inflados:", len(inflados), "| pontos:", inflados["q"].sum(), "| quando:", inflados["criado_em"].min()[:10], "a", inflados["criado_em"].max()[:10])
print("ajustes 'correcao bug app':", len(ajustes_bug), "| pontos:", ajustes_bug["q"].sum(), "| quando:", ajustes_bug["criado_em"].min()[:10], "a", ajustes_bug["criado_em"].max()[:10])
sem_ajuste = inflados[~inflados["cpf"].isin(ajustes_bug["cpf"])]
print("inflados SEM correção:", sem_ajuste["cpf"].nunique(), "clientes,", sem_ajuste["q"].sum(), "pontos")
print("ajustados SEM crédito inflado:", (~ajustes_bug["cpf"].isin(inflados["cpf"])).sum(), "clientes,", ajustes_bug.loc[~ajustes_bug["cpf"].isin(inflados["cpf"]), "q"].sum(), "pontos")

In [ ]:
# resgates anotados na planilha a partir de 2025: também estão no banco?
resgates["dia"] = pd.to_datetime(resgates["data"])
resgates_banco = pontos[pontos["tipo"] == "D"].assign(dia=lambda d: pd.to_datetime(d["criado_em"]).dt.tz_convert("America/Sao_Paulo").dt.tz_localize(None).dt.normalize())
recentes = resgates[(resgates["dia"] >= "2025-01-01") & resgates["cliente_cpf"].notna()].copy()
recentes["no_banco"] = [((resgates_banco["cpf"] == cpf) & (resgates_banco["q"] == float(q)) & ((resgates_banco["dia"] - dia).abs() <= pd.Timedelta(days=3))).any()
                        for cpf, q, dia in zip(recentes["cliente_cpf"], recentes["pontos usados"], recentes["dia"])]
print("resgates da planilha desde 2025:", (resgates["dia"] >= "2025-01-01").sum(), "| com CPF:", len(recentes), "| sem CPF (só nome):", ((resgates["dia"] >= "2025-01-01") & resgates["cliente_cpf"].isna()).sum())
recentes.groupby("loja")["no_banco"].agg(["sum", "count"])

In [ ]:
# clientes antigos (programa de 2023): quantos também estão na aba Clientes
antigos.assign(tambem_em_clientes=antigos["CPF/CNPJ"].isin(clientes["cpf"])).groupby(antigos["STATUS"].fillna("(vazio)"))["tambem_em_clientes"].agg(["sum", "count"])

## 4. Como se liga às outras fontes?

In [ ]:
print("compras por origem e loja:")
display(pd.crosstab(compras["origem"], compras["loja_id"]))
sem_pontos = compras[~compras["id"].isin(pontos["compra_id"].dropna().str.removesuffix(".0"))]
print("compras sem crédito de pontos:", len(sem_pontos), "| por tipo de cliente:", sem_pontos["cliente_tipo"].fillna("(vazio)").value_counts().to_dict())
print("CPFs com pontos no banco que não estão na planilha:", pontos.loc[~pontos["cpf"].isin(clientes["cpf"]), "cpf"].nunique())

## Desafios da revisão

In [ ]:
# nos CPFs com 2 ou mais cadastros: os dois têm pontos?
def pontos_planilha(valor):  # "2.203", "1389 pts", "15,0k" → número (só para medir)
    if pd.isna(valor):
        return 0.0
    s = str(valor).lower().replace("pts", "").strip()
    return float(s[:-1].replace(",", ".")) * 1000 if s.endswith("k") else float(s.replace(".", ""))

clientes["pontos_num"] = clientes["Pontos"].map(pontos_planilha)
duplicados = clientes[clientes["cpf"].notna() & clientes["cpf"].duplicated(keep=False)]
com_pontos = duplicados.groupby("cpf")["pontos_num"].apply(lambda s: (s > 0).sum())
print("CPFs duplicados:", len(com_pontos), "| nenhum cadastro com pontos:", (com_pontos == 0).sum(),
      "| só 1 com pontos:", (com_pontos == 1).sum(), "| 2 ou mais com pontos:", (com_pontos >= 2).sum())
print("pontos nos cadastros extras (além do maior de cada CPF):", duplicados.groupby("cpf")["pontos_num"].apply(lambda s: s.sum() - s.max()).sum())

In [ ]:
# os CPFs só no banco: clientes novos ou CPF digitado errado (1 dígito diferente de um CPF da planilha)?
# compara só com os CPFs válidos da planilha; os 28 com dígito errado ficam vazios no staging (ver a seção K2)
inicio = pontos[pontos["cpf"].notna() & ~pontos["cpf"].isin(clientes["cpf"])].groupby("cpf")["criado_em"].min()
cpfs_planilha = list(clientes["cpf"].dropna().unique())
um_digito = sum(any(sum(a != b for a, b in zip(cpf, outro)) == 1 for outro in cpfs_planilha) for cpf in inicio.index)
print("CPFs só no banco:", len(inicio), "| começaram em 2025:", (inicio < "2026").sum(), "| em 2026:", (inicio >= "2026").sum(),
      "| diferem só 1 dígito de um CPF da planilha:", um_digito)

## CPF com dígito verificador errado (K2)

O staging deixa o `cpf` vazio quando o dígito verificador não fecha; o valor digitado está em `pendencias.csv`. O CPF certo aparece em outra fonte, a 1 dígito do digitado?

In [ ]:
import re
import sys

sys.path.insert(0, str(RAIZ))
from src.utils.cpf import cpf_valido

pendencias = pd.read_csv(STAGING / "pendencias.csv", dtype=str)
vendas = pd.read_csv(STAGING / "banco_vendas.csv", dtype=str)
so_digitos = lambda s: re.sub(r"\D", "", s) if isinstance(s, str) else None
dv_errado = pendencias[(pendencias["coluna"] == "cpf") & pendencias["motivo"].str.contains("verificador")].set_index("linha")["valor"]
# o valor digitado de volta em cada linha
cpf_cadastro_errado = clientes["linha_origem"].map(dv_errado).map(so_digitos)
pontos["cpf_digitado"] = pontos["linha_origem"].map(dv_errado).map(so_digitos)

# todos os CPFs válidos que aparecem em alguma fonte
conhecidos = set()
for serie in [clientes["cpf"], pontos["cpf"], resgates["cliente_cpf"], antigos["CPF/CNPJ"], compras["cliente_ref"], vendas["cliente_ref"]]:
    conhecidos |= {c for c in serie.map(so_digitos).dropna() if cpf_valido(c)}

def candidatos(digitado):  # CPFs conhecidos que diferem em só 1 dígito
    return [c for c in conhecidos if sum(a != b for a, b in zip(c, digitado)) == 1]

print("CPFs válidos conhecidos:", len(conhecidos), "| dígito errado no cadastro:", cpf_cadastro_errado.notna().sum(),
      "| nos movimentos de pontos:", pontos["cpf_digitado"].notna().sum())

In [ ]:
# cadastro: o CPF certo aparece em outra fonte?
errados = clientes[cpf_cadastro_errado.notna()].assign(digitado=cpf_cadastro_errado.dropna())
errados["certo"] = errados["digitado"].map(candidatos).map(lambda c: c[0] if len(c) == 1 else None)
print("dígito errado no cadastro:", len(errados), "| com 1 candidato:", errados["certo"].notna().sum(),
      "| pontos na planilha:", errados.loc[errados["certo"].notna(), "pontos_num"].sum(), "| sem candidato:", errados.loc[errados["certo"].isna(), "pontos_num"].sum())
print("o certo é outro cadastro (mesma pessoa):", errados["certo"].isin(clientes["cpf"]).sum(),
      "| o certo está entre os CPFs só no banco:", errados["certo"].isin(inicio.index).sum())
errados[["linha_origem", "Nome", "Data Cadastro", "Pontos", "digitado", "certo"]]

In [ ]:
# pontos: o sistema creditava qualquer número só com dígitos, então parte dos "CPFs errados" é telefone (J1c)
DDDS = {11, 12, 13, 14, 15, 16, 17, 18, 19, 21, 22, 24, 27, 28, 31, 32, 33, 34, 35, 37, 38, 41, 42, 43, 44, 45, 46, 47, 48, 49, 51, 53, 54, 55,
        61, 62, 63, 64, 65, 66, 67, 68, 69, 71, 73, 74, 75, 77, 79, 81, 82, 83, 84, 85, 86, 87, 88, 89, 91, 92, 93, 94, 95, 96, 97, 98, 99}
telefones = set(clientes["Tel"].dropna().map(so_digitos).str[-11:])
# compra com o CPF escrito com máscara: o cliente_tipo fica vazio, mas é CPF (não celular)
com_mascara = set(compras.loc[compras["cliente_ref"].str.contains(r"\d{3}\.\d{3}\.\d{3}-\d{2}", na=False), "id"])

def grupo(digitado, compra):
    if len(set(digitado)) == 1:
        return "CPF de preenchimento"
    if digitado in telefones:
        return "telefone do cadastro (J1c)"
    if int(digitado[:2]) in DDDS and digitado[2] == "9" and compra not in com_mascara:
        return "celular fora do cadastro (J1c)"
    return "CPF digitado errado"

errados_pts = pontos[pontos["cpf_digitado"].notna()].copy()
errados_pts["mov"] = errados_pts["q"].where(errados_pts["tipo"] != "D", -errados_pts["q"])
errados_pts["grupo"] = [grupo(d, c) for d, c in zip(errados_pts["cpf_digitado"], errados_pts["compra_id"].str.removesuffix(".0"))]
print("tipo do cliente_ref na compra:", errados_pts["compra_id"].str.removesuffix(".0").map(compras.set_index("id")["cliente_tipo"]).fillna("(vazio)").value_counts().to_dict())
errados_pts.groupby("grupo").agg(movimentos=("q", "size"), numeros=("cpf_digitado", "nunique"), pontos=("mov", "sum"))

In [ ]:
# CPF digitado errado: o certo aparece em outra fonte? o cliente certo já comprou na mesma loja?
cpf_errado = errados_pts[errados_pts["grupo"] == "CPF digitado errado"].groupby("cpf_digitado")["mov"].sum().to_frame("pontos")
cpf_errado["certo"] = [c[0] if len(c) == 1 else None for c in map(candidatos, cpf_errado.index)]
lojas = compras.assign(ref=compras["cliente_ref"].map(so_digitos)).groupby("ref")["loja_id"].agg(set)
mesma_loja = [bool(lojas.get(d, set()) & lojas[c]) for d, c in zip(cpf_errado.index, cpf_errado["certo"]) if c in lojas.index]
print("números:", len(cpf_errado), "| com 1 candidato:", cpf_errado["certo"].notna().sum(), "| pontos:", cpf_errado.loc[cpf_errado["certo"].notna(), "pontos"].sum(),
      "| o certo tem movimentos próprios:", cpf_errado["certo"].isin(pontos["cpf"]).sum(), "| está no cadastro:", cpf_errado["certo"].isin(clientes["cpf"]).sum(),
      "| já comprou na mesma loja:", sum(mesma_loja), "de", len(mesma_loja))
# quantos CPFs válidos existem a 1 dígito de cada número, só pela matemática (entre ~1 bilhão de CPFs possíveis)
vizinhos = [sum(cpf_valido(d[:i] + n + d[i + 1:]) for i in range(11) for n in "0123456789" if n != d[i]) for d in cpf_errado.index]
print("CPFs válidos a 1 dígito, pela matemática: de", min(vizinhos), "a", max(vizinhos))
# celulares fora do cadastro que também batem com um CPF conhecido (seguem como telefone, J1c)
celulares = errados_pts[errados_pts["grupo"] == "celular fora do cadastro (J1c)"].groupby("cpf_digitado")["mov"].sum()
com_cpf = celulares[[len(candidatos(d)) == 1 for d in celulares.index]]
print("celulares fora do cadastro que batem com um CPF conhecido a 1 dígito:", len(com_cpf), "| pontos:", com_cpf.sum())

## Mesmo CPF em vários cadastros (K1)

O sistema lê só o primeiro cadastro com o CPF (`busca_cliente` em `planilha.py` para na primeira linha). Os cadastros do mesmo CPF são a mesma pessoa? E que pontos da planilha o sistema não mostrava?

In [ ]:
# os cadastros do mesmo CPF: mesma pessoa?
duplicados = duplicados.assign(ordem=duplicados["linha_origem"].str.split("!").str[1].astype(int)).sort_values(["cpf", "ordem"])
por_cpf = duplicados.groupby("cpf")
print("CPFs:", por_cpf.ngroups, "| cadastros por CPF:", por_cpf.size().value_counts().to_dict(),
      "| mesmo nome completo:", (por_cpf["nome_busca"].nunique() == 1).sum(),
      "| mesmo primeiro nome:", (por_cpf["nome_busca"].agg(lambda s: s.str.split().str[0].nunique()) == 1).sum())
print("mesmo telefone (quando tem):", (por_cpf["Tel"].nunique() <= 1).sum(), "| mesmo e-mail:", (por_cpf["E-mail"].agg(lambda s: s.str.lower().nunique()) == 1).sum(),
      "| mesma loja de cadastro:", (por_cpf["loja_cadastro"].agg(lambda s: s.str.lower().nunique()) == 1).sum())
anos = pd.to_datetime(duplicados["Data Cadastro"], errors="coerce").groupby(duplicados["cpf"]).agg(lambda s: (s.max() - s.min()).days / 365)
extras = duplicados[duplicados["cpf"].duplicated()]  # todos menos a primeira linha de cada CPF
print("anos entre o 1º e o último cadastro (média):", round(anos.mean(), 1), "| pontos fora do 1º cadastro:", extras["pontos_num"].sum(),
      "| CPFs com o mesmo saldo em dois cadastros:", por_cpf["pontos_num"].agg(lambda s: s[s > 0].duplicated().any()).sum())
depois_2025 = pd.to_datetime(extras["Data Cadastro"], errors="coerce") >= "2025-01-01"
print("cadastros extras feitos depois de 01/01/2025 com pontos (os 26 da K5e):", (depois_2025 & (extras["pontos_num"] > 0)).sum(), "| pontos:", extras.loc[depois_2025, "pontos_num"].sum())

In [ ]:
# o saldo da planilha que o sistema mostrava: lê a planilha original como o planilha.py (o staging já limpou o CPF)
from src.config.settings import VULTO_DATA_DIR

ORIGEM = Path(VULTO_DATA_DIR) if VULTO_DATA_DIR else RAIZ.parent / "vulto-backend" / "data"
original = pd.read_excel(ORIGEM / "clientes_fidelidade_FINAL_v3 (1).xlsx", sheet_name="Clientes", dtype=object).fillna("")
assert len(original) == len(clientes)

def parse_pontos_vulto(p):  # o parse_pontos do sistema: "2,7k" dá erro e o cliente vê 0
    try:
        if p == "":
            return 0
        if isinstance(p, (int, float)):
            return int(p)
        return int(str(p).replace("pts", "").replace(" ", "").replace(".", ""))
    except ValueError:
        return None

valido = clientes["cpf"].notna()
cpf_sistema = original["cpf"].map(lambda c: str(c).replace(".", "").replace("-", ""))  # limpa_cpf: não tira espaço
achado = valido & (cpf_sistema == clientes["cpf"])  # o login com os 11 dígitos acha esta linha
mostrado = achado & ~clientes["cpf"].where(achado).dropna().duplicated().reindex(clientes.index, fill_value=False)  # a primeira achada
print("CPF que o login não acha:", (valido & ~achado).sum(), "| formas:", cpf_sistema[valido & ~achado].str.replace(r"\d", "9", regex=True).value_counts().to_dict())

motivo = pd.Series("aparece", index=clientes.index)
motivo[mostrado & original["Pontos"].map(parse_pontos_vulto).isna()] = "abreviado vira 0"
motivo[valido & ~mostrado & clientes["cpf"].isin(clientes.loc[mostrado, "cpf"])] = "outro cadastro do CPF é o que aparece"
motivo[valido & ~mostrado & ~clientes["cpf"].isin(clientes.loc[mostrado, "cpf"])] = "login não acha nenhum cadastro do CPF"
print("o sistema não mostrava:", clientes.loc[valido & (motivo != "aparece"), "pontos_num"].sum(), "de", clientes.loc[valido, "pontos_num"].sum(), "pts da planilha (CPF válido)")
clientes[valido].groupby(motivo[valido]).agg(cadastros=("pontos_num", "size"), com_pontos=("pontos_num", lambda s: (s > 0).sum()), pontos=("pontos_num", "sum"))

## Resgates só na planilha (K7)

Algumas lojas anotaram resgates na planilha que não estão no banco. Eles aconteceram (a venda tem o desconto)? E os que só têm o nome do cliente?

In [ ]:
# quantidade com ponto de milhar ("2.000"): o padronizar.py leu como 2; o desconto em R$ confirma o milhar
original_resgates = pd.read_excel(ORIGEM / "clientes_fidelidade_FINAL_v3 (1).xlsx", sheet_name="Resgates", dtype=object)
assert len(original_resgates) == len(resgates)
pontos_texto = pd.Series(original_resgates["pontos usados"].astype(str).values, index=resgates.index)
milhar = pontos_texto.str.match(r"^\d+\.\d{3}$")
resgates["pts"] = pd.to_numeric(resgates["pontos usados"]).where(~milhar, pontos_texto.str.replace(".", "", regex=False).astype(float))
resgates["desconto"] = pd.to_numeric(resgates["desconto R$"])
print("com ponto de milhar:", milhar.sum(), "| R$ por ponto neles:", (resgates.loc[milhar, "desconto"] / resgates.loc[milhar, "pts"]).round(2).value_counts().to_dict())

In [ ]:
# resgate da planilha desde 2025 × banco (mesmo CPF e quantidade, até 3 dias) × venda (mesmo desconto, loja e CPF, até 1 dia)
LOJAS = {"Pinheiros": "1", "Eldorado": "2", "Savassi": "3", "Batel": "4"}
vendas["dia"] = pd.to_datetime(vendas["criado_em"]).dt.tz_convert("America/Sao_Paulo").dt.tz_localize(None).dt.normalize()
vendas["desconto_num"], vendas["ref"] = pd.to_numeric(vendas["desconto"]), vendas["cliente_ref"].map(so_digitos)

def no_banco(cpf, q, dia, dias=3):
    return ((resgates_banco["cpf"] == cpf) & (resgates_banco["q"] == q) & ((resgates_banco["dia"] - dia).dt.days.abs() <= dias)).any()

def vendas_com_desconto(loja, desconto, dia):
    return vendas[(vendas["loja_id"] == LOJAS.get(loja)) & ((vendas["dia"] - dia).dt.days.abs() <= 1) & ((vendas["desconto_num"] - desconto).abs() < 0.01)]

def tem_venda(linhas):
    return [(vendas_com_desconto(l, ds, d)["ref"] == c).any() for l, ds, d, c in zip(linhas["loja"], linhas["desconto"], linhas["dia"], linhas["cliente_cpf"])]

desde_2025 = resgates[resgates["dia"] >= "2025-01-01"]
com_cpf = desde_2025[desde_2025["cliente_cpf"].notna()]
so_planilha = com_cpf[[not no_banco(c, q, d) for c, q, d in zip(com_cpf["cliente_cpf"], com_cpf["pts"], com_cpf["dia"])]]
loja_no_banco = com_cpf[(com_cpf["loja"] != "Site") & ~com_cpf.index.isin(so_planilha.index)]
print("com CPF:", len(com_cpf), "| no banco:", len(com_cpf) - len(so_planilha), "| só na planilha:", len(so_planilha),
      "| pontos:", so_planilha["pts"].sum(), "| desconto R$:", so_planilha["desconto"].sum())
print("têm a venda com o desconto | só na planilha:", sum(tem_venda(so_planilha)), "de", len(so_planilha),
      "| de loja e no banco (comparação):", sum(tem_venda(loja_no_banco)), "de", len(loja_no_banco))

In [ ]:
# só com o nome: a venda com o mesmo desconto dá o CPF, e o nome do cadastro desse CPF confirma (o nome sozinho não liga)
from migracao_vulto.normalizadores import chave_busca

nomes_cpf = clientes.dropna(subset=["cpf"]).groupby("cpf")["nome_busca"].agg(set)

def confirma(nome, cpf):  # nome completo: igual ao do cadastro; só o primeiro nome: igual ao primeiro nome do cadastro
    return any(n == nome if " " in nome else n.split()[0] == nome for n in nomes_cpf.get(cpf, set()))

sem_cpf = desde_2025[desde_2025["cliente_cpf"].isna()].copy()
sem_cpf["nome"] = sem_cpf["cliente_nome"].map(chave_busca)
sem_cpf["achados"] = [{c for c in vendas_com_desconto(l, ds, d)["ref"].dropna() if confirma(n, c)}
                      for n, l, ds, d in zip(sem_cpf["nome"], sem_cpf["loja"], sem_cpf["desconto"], sem_cpf["dia"])]
sem_cpf["cliente_cpf"] = sem_cpf["achados"].map(lambda s: next(iter(s)) if len(s) == 1 else None)
achado = sem_cpf["cliente_cpf"].notna()
print("só com o nome:", len(sem_cpf), "| pontos:", sem_cpf["pts"].sum(), "| acham 1 CPF pela venda:", achado.sum(),
      "(nome completo:", (achado & sem_cpf["nome"].str.contains(" ")).sum(), ") | pontos:", sem_cpf.loc[achado, "pts"].sum(),
      "| sem cliente:", (~achado).sum(), sem_cpf.loc[~achado, "pts"].sum())
print("o CPF achado já tem esse resgate no banco (até 31 dias):",
      sum(no_banco(c, q, d, dias=31) for c, q, d in zip(sem_cpf.loc[achado, "cliente_cpf"], sem_cpf.loc[achado, "pts"], sem_cpf.loc[achado, "dia"])))

## Saldo inicial (K5)

De onde vem o saldo de cada cliente: a planilha congelada em 01/01/2025, o banco depois disso, e as duas abas da planilha (antiga e nova).

In [ ]:
# o banco continua o saldo da planilha? clientes que ficam negativos com e sem o saldo inicial
pontos["mov"] = pontos["q"].where(pontos["tipo"] != "D", -pontos["q"])  # D vem positivo e subtrai
inicial = clientes.dropna(subset=["cpf"]).groupby("cpf")["pontos_num"].sum()
ordem = pontos[pontos["cpf"].isin(inicial.index)].sort_values("criado_em")
menor = ordem.groupby("cpf")["mov"].cumsum().groupby(ordem["cpf"]).min()
print("negativos em algum momento | só banco:", (menor < 0).sum(), "| planilha + banco:", (menor + inicial.reindex(menor.index) < 0).sum())
print("saldo da planilha (CPF válido):", inicial.sum(), "| banco líquido:", pontos["mov"].sum())

In [ ]:
# saldos abreviados ("2,7k") e o teto de 15.000
abreviados = clientes["Pontos"].fillna("").str.lower().str.endswith("k")
print("abreviados:", abreviados.sum(), "| erro máximo: ±", abreviados.sum() * 50, "pts")
pd.cut(clientes["pontos_num"], [12999, 13999, 14999, 15000, float("inf")]).value_counts().sort_index()

In [ ]:
# clientes antigos: nas duas abas (180) e só na antiga (120)
# com o CPF corrigido pela K2: quem está na aba nova com o dígito errado também está nas duas abas
cpf_k2 = clientes["cpf"].fillna(clientes["linha_origem"].map(errados.set_index("linha_origem")["certo"]))
antigos["p2023"] = pd.to_numeric(antigos["PONTOS_2023"])
saldo_2025 = clientes.assign(cpf=cpf_k2).dropna(subset=["Pontos"]).groupby("cpf")["pontos_num"].sum()  # Pontos vazio fica de fora, não vira 0
nas_duas = antigos[antigos["CPF/CNPJ"].isin(cpf_k2)].assign(p2025=lambda d: d["CPF/CNPJ"].map(saldo_2025))
print("nas duas abas:", len(nas_duas), "| 2023:", nas_duas["p2023"].sum(), "| 2025:", nas_duas["p2025"].sum(),
      "| caiu:", (nas_duas["p2025"] < nas_duas["p2023"]).sum(), "| subiu:", (nas_duas["p2025"] > nas_duas["p2023"]).sum(),
      "| vazio em 2025:", nas_duas["p2025"].isna().sum())
# a queda não segue a data da última compra (se fosse vencimento, quem comprou antes perderia mais)
display((nas_duas["p2025"] / nas_duas["p2023"]).groupby(pd.to_datetime(nas_duas["ULT_COMPRA"]).dt.quarter).median())
so_antigos = antigos[~antigos["CPF/CNPJ"].isin(cpf_k2)]
voltaram = pontos[pontos["cpf"].isin(so_antigos["CPF/CNPJ"])]
print("só na antiga:", len(so_antigos), "| pontos:", so_antigos["p2023"].sum(),
      "| ATIVO:", (so_antigos["STATUS"] == "ATIVO").sum(), so_antigos.loc[so_antigos["STATUS"] == "ATIVO", "p2023"].sum(),
      "| voltaram em 2025-26:", voltaram["cpf"].nunique(), "| resgataram:", voltaram.loc[voltaram["tipo"] == "D", "cpf"].nunique())

In [ ]:
# cadastrados depois do congelamento, mas com pontos na planilha
depois = clientes[(pd.to_datetime(clientes["Data Cadastro"], errors="coerce") >= "2025-01-01") & (clientes["pontos_num"] > 0)]
online = depois["loja_cadastro"].str.lower().isin(["app", "site"])
compras_deles = compras[compras["cliente_ref"].isin(depois["cpf"])]
print("cadastrados depois de 2025 com pontos:", len(depois), "| pontos:", depois["pontos_num"].sum(), "| pelo site ou app:", online.sum(),
      "| compras sem crédito:", (~compras_deles["id"].isin(pontos["compra_id"].dropna().str.removesuffix(".0"))).sum())

In [ ]:
# saldo negativo mesmo com o saldo inicial: volta logo, volta depois ou nunca volta
ordem = ordem.assign(saldo=ordem.groupby("cpf")["mov"].cumsum() + ordem["cpf"].map(inicial), quando=pd.to_datetime(ordem["criado_em"]))

def recuperacao(d):
    primeiro_negativo = d.loc[d["saldo"] < 0, "quando"].iloc[0]
    volta = d[(d["quando"] > primeiro_negativo) & (d["saldo"] >= 0)]
    if volta.empty:
        return "nunca volta"
    return "volta em menos de 1h" if volta["quando"].iloc[0] - primeiro_negativo < pd.Timedelta(hours=1) else "volta depois"

negativos = ordem[ordem["cpf"].isin(ordem.loc[ordem["saldo"] < 0, "cpf"])]
grupos = pd.Series({cpf: recuperacao(d) for cpf, d in negativos.groupby("cpf")})
print(grupos.value_counts().to_dict(), "| saldo final dos que nunca voltam:",
      negativos[negativos["cpf"].isin(grupos[grupos == "nunca volta"].index)].groupby("cpf")["saldo"].last().sum())

## CPFs só nos pontos (K8)

CPFs com movimentos no banco que não estão em nenhuma aba da planilha (fora os 24 da K2). São clientes de verdade? Há nome ou e-mail em alguma fonte?

In [ ]:
# CPFs que só existem nos pontos (fora da K2): quem são, quanto devem, e há nome ou e-mail em outra fonte?
so_pontos = sorted(set(pontos["cpf"].dropna()) - set(clientes["cpf"].dropna()) - set(antigos["CPF/CNPJ"]))
mov_so = pontos[pontos["cpf"].isin(so_pontos)]
saldo_so = mov_so.groupby("cpf")["mov"].sum()
compras_so = compras[compras["id"].isin(mov_so["compra_id"].dropna().str.replace(".0", "", regex=False))]
print("CPFs:", len(so_pontos), "| válidos:", sum(map(cpf_valido, so_pontos)), "| pontos:", saldo_so.sum(), "| negativos:", saldo_so[saldo_so < 0].to_dict() and saldo_so[saldo_so < 0].tolist())
print("1º movimento por ano:", pd.to_datetime(mov_so.groupby("cpf")["criado_em"].min(), utc=True).dt.year.value_counts().to_dict(),
      "| compras por origem:", compras_so["origem"].value_counts().to_dict())
print("nas vendas:", vendas["cliente_valor"].isin(so_pontos).sum(), "vendas de", vendas.loc[vendas["cliente_valor"].isin(so_pontos), "cliente_valor"].nunique(), "CPFs",
      "| nos resgates da planilha com nome:", resgates.loc[resgates["cliente_cpf"].isin(so_pontos), "cliente_nome"].notna().sum())
# o único negativo: CPF clássico de teste, pontos só de vendas de teste (V8)
teste = mov_so[saldo_so.reindex(mov_so["cpf"]).lt(0).values]
vendas_teste = vendas[vendas["compra_id"].isin(teste["compra_id"].dropna())]
print("negativo:", teste["cpf"].unique(), "| movimentos:", teste["tipo"].value_counts().to_dict(), "| pedidos:", vendas_teste["pedido_externo"].str[:6].unique(), "| obs:", vendas_teste["obs"].dropna().unique())

## Pontos de vendas que não valem e bug do app (K6)

O sistema credita na hora ("o sistema credita na hora, TODO") e a regra manda estornar o ponto de compra cancelada. Quanto ponto veio de venda que não vale pelas decisões V3, V4, V5 e V8, ou do bug do app, e ainda não foi estornado?

In [ ]:
# quais vendas não valem (V3, V4, V5, V8) e o valor em reais (V1: o app grava parte em centavos)
itens_venda = pd.read_csv(STAGING / "banco_itens_venda.csv", dtype=str)
catalogo = pd.read_csv(STAGING / "catalogo_produtos.csv", dtype=str)

def chave_sku(sku):  # o mesmo SKU escrito de jeitos diferentes (como no notebook 01)
    s = str(sku).upper().strip()
    longo = re.fullmatch(r"VLT[- ]([A-Z]{3})[- ](\d{3})[- ]([A-Z]+)[- ]?(.*)", s)
    if longo:
        return f"{longo[1]}{longo[2]}{longo[3][0]}{longo[4].replace(' ', '').replace('-', '')}"
    curto = re.fullmatch(r"([A-Z]{3})(\d{3})([A-Z])(.+)", s)
    return f"{curto[1]}{curto[2]}{curto[3]}{curto[4]}" if curto else None

preco = catalogo.assign(k=catalogo["sku"].map(chave_sku), p=pd.to_numeric(catalogo["preco"])).groupby("k")["p"].max()
escala = (pd.to_numeric(itens_venda["preco_unit"]) / itens_venda["sku"].map(chave_sku).map(preco)).groupby(itens_venda["venda_id"]).median()
vendas["t"] = pd.to_numeric(vendas["total"])
centavos = vendas["id"].map(escala).gt(50) | (vendas["id"].map(escala).isna() & vendas["t"].abs().gt(3000))
vendas["reais"] = vendas["t"].where(~centavos, vendas["t"] / 100)
vendas["quando"] = pd.to_datetime(vendas["criado_em"], utc=True)
situacao = vendas["status"].str.lower()
email = vendas["cliente_valor"].fillna("").str.lower()
teste = vendas["obs"].fillna("").str.lower().eq("teste") | email.str.startswith("teste@") | email.eq("qa.app@vulto.com.br") | vendas["cliente_valor"].eq("12345678909")
repetidos = vendas[vendas["pedido_externo"].notna() & vendas["pedido_externo"].duplicated(keep=False)].sort_values(["pedido_externo", "quando"])
copia = vendas["id"].isin(repetidos.loc[repetidos["pedido_externo"].duplicated(keep="last"), "id"])
vendas["grupo"] = "vale"
vendas.loc[situacao.isin(["estornado", "chargeback"]) & vendas["t"].gt(0), "grupo"] = "estornado/chargeback (V4)"
vendas.loc[situacao.eq("aguardando"), "grupo"] = "aguardando (V3)"
vendas.loc[situacao.isin(["cancelado", "cancelada"]), "grupo"] = "cancelada"
vendas.loc[copia, "grupo"] = "cópia (V5)"
vendas.loc[teste, "grupo"] = "teste (V8)"
print(vendas["grupo"].value_counts().to_dict())

In [ ]:
# créditos de pontos de vendas que não valem e do bug do app, o que já foi estornado e o que falta
creditos = pontos[pontos["tipo"] == "C"].assign(cid=lambda d: d["compra_id"].str.removesuffix(".0")).merge(
    compras[["id", "v", "origem", "cliente_ref", "loja_id", "criado_em"]], left_on="cid", right_on="id", suffixes=("", "_compra"))
grupo_da_compra = vendas.dropna(subset=["compra_id"]).assign(cid=lambda d: d["compra_id"].str.removesuffix(".0")).set_index("cid")["grupo"]
creditos["grupo"] = creditos["cid"].map(grupo_da_compra)
# loja física: a compra não aponta para a venda; liga pelo cliente, valor, loja e dia
fisica = vendas[vendas["loja_id"].isin(["1", "2", "3", "4"]) & vendas["grupo"].eq("cancelada")]
chave_fisica = set(zip(fisica["cliente_ref"], fisica["t"], fisica["loja_id"], fisica["quando"].dt.date))
dia_compra = pd.to_datetime(creditos["criado_em_compra"], utc=True).dt.date
na_fisica = pd.Series([k in chave_fisica for k in zip(creditos["cliente_ref"], creditos["v"], creditos["loja_id"], dia_compra)], index=creditos.index)
creditos.loc[creditos["origem"].eq("pdv") & na_fisica, "grupo"] = "cancelada (loja física)"
estornos = pontos[pontos["tipo"] == "E"].assign(cid=lambda d: d["compra_id"].str.removesuffix(".0")).groupby("cid")["q"].sum()
creditos["ja_estornado"] = creditos["cid"].map(estornos).fillna(0)

# bug do app (08–12/06/2026): crédito sobre o valor em centavos; a Vulto corrigiu parte deixando 1% do crédito
creditos["inflado"] = creditos["origem"].eq("app") & creditos["v"].gt(3000) & (creditos["q"] / creditos["v"]).ge(0.9)
corrigidos = set(pontos.loc[(pontos["tipo"] == "AJ") & pontos["obs"].fillna("").str.contains("bug"), "cpf"])
sem_bug = creditos["q"] - (creditos["q"] / 100).round()
inflados = creditos[creditos["inflado"]]
print("inflados:", len(inflados), "| pontos:", inflados["q"].sum(), "| quando:", inflados["criado_em"].str[:10].min(), "a", inflados["criado_em"].str[:10].max(),
      "| clientes corrigidos:", inflados.loc[inflados["cpf"].isin(corrigidos), "cpf"].nunique(), "| sem correção:", inflados.loc[~inflados["cpf"].isin(corrigidos), "cpf"].nunique())

invalido = creditos["grupo"].isin(["aguardando (V3)", "cancelada", "cópia (V5)", "estornado/chargeback (V4)", "cancelada (loja física)"])
ajuste_feito = sem_bug.where(creditos["inflado"] & creditos["cpf"].isin(corrigidos), 0)
creditos["falta"] = 0.0
creditos.loc[invalido, "falta"] = creditos["q"] + creditos["ja_estornado"] - ajuste_feito
creditos.loc[~invalido & creditos["inflado"] & ~creditos["cpf"].isin(corrigidos), "falta"] = sem_bug
creditos["motivo"] = creditos["grupo"].where(invalido, "bug do app sem correção")
estornar = creditos[creditos["falta"] > 0]
display(estornar.groupby("motivo").agg(creditos=("id", "size"), clientes=("cpf", "nunique"), falta=("falta", "sum")))
antigo = pd.to_datetime(estornar["criado_em"], utc=True) < pd.Timestamp("2025-03-01", tz="UTC")
print("total:", len(estornar), "créditos,", estornar["cpf"].nunique(), "clientes,", estornar["falta"].sum(), "pts | R$ pela K3:",
      round(estornar.loc[antigo, "falta"].sum() * 0.10 + estornar.loc[~antigo, "falta"].sum() * 0.05, 2))

# alguém fica negativo? saldo da planilha + banco, menos o estorno
saldo = pontos[pontos["cpf"].ne("12345678909")].groupby("cpf")["mov"].sum().add(inicial, fill_value=0)
depois = saldo.sub(estornar.groupby("cpf")["falta"].sum(), fill_value=0)
print("ficam negativos por causa do estorno:", ((depois < 0) & (saldo >= 0)).sum())

## Valor do ponto (K3)

A LEIA ISSO diz que 100 pontos valem R$ 5 desde mar/2025 (eram R$ 10), mas o sistema continua com `valor_ponto = 0,10`. Quanto a Vulto pagou em cada época, e quanto vale o saldo se o ponto ganho até fev/2025 valer R$ 0,10 e o ganho depois, R$ 0,05?

In [ ]:
# quanto a Vulto pagou por ponto em cada resgate da planilha: o sistema (/resgate) usa valor_ponto = 0,10; a loja à mão, a regra
resgates["dia"] = pd.to_datetime(resgates["data"], errors="coerce")
resgates["por_ponto"] = (resgates["desconto"] / resgates["pts"]).round(2)
depois = resgates[resgates["dia"] >= "2025-03-01"]
print("até fev/2025:", resgates.loc[resgates["dia"] < "2025-03-01", "por_ponto"].value_counts().to_dict())
display(pd.crosstab([depois["loja"], depois["cliente_cpf"].notna().rename("com CPF")], depois["por_ponto"]))

In [ ]:
# passivo em R$ pela regra da K3: ponto ganho até fev/2025 vale R$ 0,10, depois R$ 0,05
# saldo = planilha (K5a, K1) + 30 ATIVOS da aba antiga (K5d) + banco; o resgate gasta primeiro os pontos mais antigos (FIFO)
CORTE = pd.Timestamp("2025-03-01", tz="UTC")
ativos = so_antigos[so_antigos["STATUS"] == "ATIVO"].set_index("CPF/CNPJ")["p2023"]
antes = inicial.add(ativos, fill_value=0)
movs = pontos[pontos["cpf"].notna()].assign(quando=pd.to_datetime(pontos["criado_em"], utc=True)).sort_values("quando")

def separa(mais_novos_primeiro=False):
    saldos = {}
    por_cpf = dict(tuple(movs.groupby("cpf")))
    for cpf in set(antes.index) | set(por_cpf):
        pote = {"antigo": antes.get(cpf, 0.0), "novo": 0.0}
        if cpf in por_cpf:
            for quando, mov in zip(por_cpf[cpf]["quando"], por_cpf[cpf]["mov"]):
                if mov >= 0:
                    pote["antigo" if quando < CORTE else "novo"] += mov
                    continue
                for qual in (["novo", "antigo"] if mais_novos_primeiro else ["antigo", "novo"]):
                    gasto = min(pote[qual], -mov)
                    pote[qual] -= gasto
                    mov += gasto
        saldos[cpf] = pote
    return pd.DataFrame(saldos).T

for nome, ordem_gasto in [("antigos primeiro (FIFO)", False), ("novos primeiro", True)]:
    s = separa(ordem_gasto)
    antigo, novo = s["antigo"].sum(), s["novo"].sum()
    print(nome, "| pts antigos:", round(antigo), "| novos:", round(novo), "| R$ pela K3:", round(antigo * 0.10 + novo * 0.05),
          "| tudo a 0,10:", round((antigo + novo) * 0.10), "| tudo a 0,05:", round((antigo + novo) * 0.05))

## Telefone sem DDD e consentimento (K9)

O `padronizar.py` apaga o telefone sem DDD e o manda para as pendências. Dá para achar o DDD em outra fonte, sem chutar? E o que quer dizer cada valor do `aceita_msg`?

In [ ]:
# telefone sem DDD: o número completo da mesma pessoa aparece em outra fonte?
import unicodedata

atendimentos = pd.read_csv(STAGING / "atendimentos.csv", dtype=str)
sem_ddd = pendencias[(pendencias["coluna"] == "Tel") & pendencias["motivo"].str.contains("DDD")].set_index("linha")["valor"]
print("sem DDD:", len(sem_ddd), "| formatos:", sem_ddd.str.replace(r"\d", "9", regex=True).value_counts().to_dict())

completos = {}  # 9 últimos dígitos → (DDD, fonte) de cada número completo
def guarda(numeros, fonte):
    for d in numeros.map(so_digitos).dropna():
        d = d[2:] if len(d) in (12, 13) and d.startswith("55") else d
        if len(d) in (10, 11):
            completos.setdefault(d[-9:], set()).add((d[:-9], fonte))

guarda(clientes["Tel"], "cadastro")
guarda(atendimentos.loc[atendimentos["contato_tipo"] == "telefone", "contato_valor"], "atendimento")
for nome, tabela in [("vendas", vendas), ("compras", compras)]:
    # "numero" só conta quando não é CPF válido (J1b: o ambíguo vira CPF)
    celular = (tabela["cliente_tipo"] == "telefone") | ((tabela["cliente_tipo"] == "numero") & ~tabela["cliente_valor"].fillna("").map(cpf_valido))
    guarda(tabela.loc[celular, "cliente_valor"], nome)

tel = clientes[clientes["linha_origem"].isin(sem_ddd.index)].copy()
tel["nove"] = tel["linha_origem"].map(sem_ddd).map(so_digitos)
tel["ddds"] = tel["nove"].map(lambda n: {ddd for ddd, _ in completos.get(n, set())})
tel["fontes"] = tel["nove"].map(lambda n: ", ".join(sorted({f for _, f in completos.get(n, set())})))
print("achados em outra fonte:", (tel["ddds"].str.len() > 0).sum(), "| com um DDD só:", (tel["ddds"].str.len() == 1).sum(),
      "| coincidência esperada:", round(len(completos) / 1e8 * len(tel), 2), f"({len(completos)} números completos em 100 milhões de celulares possíveis)")
print(tel.loc[tel["fontes"] != "", "fontes"].value_counts().to_dict())

In [ ]:
# os achados são a mesma pessoa? outro cadastro com o mesmo número: mesmo CPF ou primeiro nome; atendimento: mesmo primeiro nome
primeiro = lambda s: unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower().split()[0].strip(".") if isinstance(s, str) and s.strip() else None
clientes["nove"] = clientes["Tel"].map(so_digitos).str[-9:]
pares = tel[tel["fontes"].str.contains("cadastro")].merge(clientes[["nove", "cpf", "Nome"]], on="nove", suffixes=("", "_outro"))
por_linha = pares.assign(igual=pares["cpf"] == pares["cpf_outro"]).groupby("linha_origem").agg(
    mesmo_cpf=("igual", "any"), nome=("Nome", "first"), outros=("Nome_outro", list))
por_linha["mesmo_nome"] = [primeiro(n) in {primeiro(o) for o in outros} for n, outros in zip(por_linha["nome"], por_linha["outros"])]
print("em outro cadastro:", len(por_linha), "| mesmo CPF:", por_linha["mesmo_cpf"].sum(), "| mesmo primeiro nome:", por_linha["mesmo_nome"].sum())

atendimentos["nove"] = atendimentos["contato_valor"].where(atendimentos["contato_tipo"] == "telefone").map(so_digitos).str[-9:]
nomes_atend = atendimentos[atendimentos["nome_contato"].map(primeiro).notna() & ~atendimentos["nome_contato"].map(primeiro).isin(["?", "cliente"])]
nomes_atend = nomes_atend.groupby("nove")["nome_contato"].agg(lambda s: {primeiro(n) for n in s})
com_nome = tel[tel["nove"].isin(nomes_atend.index)]
print("no atendimento com nome:", len(com_nome), "| mesmo primeiro nome:", sum(primeiro(n) in nomes_atend[v] for n, v in zip(com_nome["Nome"], com_nome["nove"])))

In [ ]:
# e o DDD da cidade da loja? confere nos achados de loja física
DDD_LOJA = {"pinheiros": "11", "eldorado": "11", "eldorado shop": "11", "batel": "41", "cwb": "41", "bh": "31", "savassi": "31"}
tel["ddd_loja"] = tel["loja_cadastro"].str.lower().str.strip().map(DDD_LOJA)
confere = tel[(tel["ddds"].str.len() == 1) & tel["ddd_loja"].notna()]
print("DDD da loja = DDD achado:", (confere["ddds"].map(min) == confere["ddd_loja"]).sum(), "de", len(confere))
resto = tel[tel["ddds"].str.len() == 0]
print("sem DDD achado:", len(resto), "| loja física:", resto["ddd_loja"].notna().sum(), "| online:", resto["ddd_loja"].isna().sum())

In [ ]:
# pontos creditados em telefone (J1c, J1d): com os 425 sem DDD, quem ainda fica sem dono?
nove_sem = set(tel["nove"])
por_grupo = errados_pts.groupby("grupo")["mov"].agg(["count", "sum"])
print("pontos com o CPF inválido, por grupo:", por_grupo.to_dict("index"))
celulares_j1c = errados_pts[errados_pts["grupo"] == "celular fora do cadastro (J1c)"].groupby("cpf_digitado")["mov"].agg(["count", "sum"])
sem_ddd_j1c = celulares_j1c.index.str[-9:].isin(nove_sem)
a_1_digito = celulares_j1c.index.to_series().map(lambda n: len(candidatos(n)) == 1).to_numpy() & ~sem_ddd_j1c
print("J1c, celular fora do cadastro:", len(celulares_j1c), "números,", celulares_j1c["count"].sum(), "créditos,", celulares_j1c["sum"].sum(), "pts",
      "| um dos 425 sem DDD:", sem_ddd_j1c.sum(), celulares_j1c.loc[sem_ddd_j1c, "sum"].sum(),
      "| a 1 dígito de um CPF conhecido (K2):", a_1_digito.sum(), celulares_j1c.loc[a_1_digito, "sum"].sum(),
      "| sem dono:", (~sem_ddd_j1c & ~a_1_digito).sum())

curto = pendencias[(pendencias["fonte"] == "vulto.db") & pendencias["motivo"].str.contains("sem 11")].set_index("linha")["valor"].map(so_digitos)
j1d = pontos[pontos["linha_origem"].isin(curto.index)].assign(nove=lambda p: p["linha_origem"].map(curto))
completo = j1d["nove"].isin(clientes["nove"].dropna())
print("J1d, celular sem DDD:", j1d["nove"].nunique(), "números,", len(j1d), "créditos,", j1d["q"].sum(), "pts",
      "| telefone completo do cadastro:", j1d.loc[completo, "nove"].nunique(), j1d.loc[completo, "q"].sum(),
      "| um dos 425 sem DDD:", j1d.loc[j1d["nove"].isin(nove_sem), "nove"].nunique(), j1d.loc[j1d["nove"].isin(nove_sem), "q"].sum(),
      "| sem dono:", j1d.loc[~completo & ~j1d["nove"].isin(nove_sem), "nove"].nunique())

In [ ]:
# consentimento: o "x" é sim ou não? nos CPFs com mais de um cadastro, quais respostas aparecem juntas
original_msg = original["aceita_msg"].astype(str).str.strip()
resposta = original_msg.map({"sim": "sim", "S": "sim", "1": "sim", "x": "x", "nao": "nao", "N": "nao", "0": "nao"}).fillna("vazio")
nao_ligar = original["Obs"].astype(str).str.strip().eq("não ligar")
print(pd.crosstab(resposta, nao_ligar.rename("Obs 'não ligar'"), margins=True))
juntas = resposta.groupby(clientes["cpf"]).agg(lambda s: " + ".join(sorted(set(s))))
print("respostas juntas no mesmo CPF:", juntas[juntas.str.contains(r"\+")].value_counts().to_dict())

# com a regra da K9b: x = sim, "não ligar" = recusa, por pessoa a recusa vale sobre o aceite e a resposta sobre o vazio
final = resposta.replace("x", "sim").mask(nao_ligar, "nao")
por_cpf = final.groupby(clientes["cpf"]).agg(lambda s: "nao" if "nao" in set(s) else "sim" if "sim" in set(s) else "não informado")
print("por pessoa:", len(por_cpf), por_cpf.value_counts().to_dict(),
      "| aceite e recusa no mesmo CPF:", final.groupby(clientes["cpf"]).agg(lambda s: {"sim", "nao"} <= set(s)).sum())

## Vencimento, Ouro e dobro no online (K4)

Regras da aba "LEIA ISSO" (jun/2025): site e app valem **dobro**; Ouro ganha **+50%** "em cima de tudo"; pontos **vencem em 12 meses** (18 no Ouro, "ainda não tá no sistema"); categoria: **Bronze até 1.999 pts acumulados no ano, Prata 2.000–4.999, Ouro 5.000+** ("na prática a categoria foi colocada na mão, não bate com essa regra").

No sistema da Vulto (`vulto-backend`): o dobro só vale quando o canal é exatamente `"site"` ou `"app"` (`CANAIS_ONLINE` em `main.py`); `config.validade_dias = 365`, mas `calcula_saldo` só tem `# TODO expirar pontos`; o Ouro não aparece no código. A tela do cliente (`vulto-frontend/src/App.jsx`) mostra o saldo sem validade e `Categoria: {dados.categoria || 'Bronze'}`.

### Dobro no online (K4b)

In [ ]:
canal_compra = compras[["id", "canal"]].rename(columns={"id": "id_compra", "canal": "canal_compra"})
cred_canal = creditos.merge(canal_compra, on="id_compra", how="left")
# pontos por real; abaixo de 0,6 o valor da compra ainda está em centavos (V1)
pontos_por_real = cred_canal["q"] / cred_canal["v"]
cred_canal["com_dobro"] = pontos_por_real.where(pontos_por_real >= 0.6, pontos_por_real * 100).round(1) >= 1.9
pd.crosstab([cred_canal["origem"], cred_canal["canal_compra"]], cred_canal["com_dobro"])

In [ ]:
# online sem dobro: falta o mesmo número de pontos do crédito; os que a K6 estorna não ganham a diferença
VALOR_K3 = lambda quando: 0.10 if quando < pd.Timestamp("2025-03-01", tz="UTC") else 0.05
sem_dobro = cred_canal[cred_canal["origem"].isin(["site", "app"]) & ~cred_canal["com_dobro"]].copy()
sem_dobro["quando"] = pd.to_datetime(sem_dobro["criado_em"], utc=True)
sem_dobro["reais"] = sem_dobro["q"] * sem_dobro["quando"].map(VALOR_K3)
na_k6 = sem_dobro["id"].isin(estornar["id"])
for nome, parte in [("todos", sem_dobro), ("sem os estornados pela K6", sem_dobro[~na_k6])]:
    print(nome, "| créditos:", len(parte), "| clientes:", parte["cpf"].nunique(), "| pontos a creditar:", round(parte["q"].sum()),
          "| R$ pela K3:", round(parte["reais"].sum(), 2))
sem_dobro[~na_k6].groupby("origem")["q"].agg(["count", "sum"])

### Vencimento (K4a)

Se o vencimento tivesse sido aplicado: cada crédito vira um lote que vence 12 meses depois; o saldo de 01/01/2025 (planilha + 30 ATIVOS da aba antiga, K5) é um lote dessa data; o débito gasta os lotes mais antigos ainda válidos. Créditos do bug do app com 1% (K6).

In [ ]:
DATA_BASE = pd.Timestamp("2026-10-01", tz="UTC")
lotes_mov = movs.copy()
lotes_mov["mov"] = lotes_mov["mov"].astype(float)
bug = lotes_mov["id"].isin(inflados["id"])
lotes_mov.loc[bug, "mov"] = lotes_mov.loc[bug, "mov"] * 0.01

def vencer(meses_por_cpf):
    resultado = {}
    por_cpf = dict(tuple(lotes_mov.sort_values("quando").groupby("cpf")))
    for cpf in set(antes.index) | set(por_cpf):
        meses = meses_por_cpf(cpf)
        lotes = [[pd.Timestamp("2025-01-01", tz="UTC"), float(antes[cpf])]] if antes.get(cpf, 0) > 0 else []
        conta = {"vencido": 0.0, "acima_do_valido": 0.0}

        def vence_ate(data):
            for lote in lotes:
                if lote[1] > 0 and lote[0] + pd.DateOffset(months=meses) <= data:
                    conta["vencido"] += lote[1]
                    lote[1] = 0

        if cpf in por_cpf:
            for quando, mov in zip(por_cpf[cpf]["quando"], por_cpf[cpf]["mov"]):
                vence_ate(quando)
                if mov >= 0:
                    lotes.append([quando, mov])
                    continue
                gastar = -mov
                for lote in lotes:
                    usado = min(lote[1], gastar)
                    lote[1] -= usado
                    gastar -= usado
                conta["acima_do_valido"] += gastar  # débito que só cabia com ponto vencido
        vence_ate(DATA_BASE)
        resultado[cpf] = conta
    return pd.DataFrame(resultado).T

print("saldo de 01/01/2025:", round(antes.sum()), "pts (venceria todo em 01/01/2026)")
print("movimentos do tipo vencimento no banco:", pontos["obs"].fillna("").str.contains("venc|expir", case=False).sum())
for nome, meses in [("12 meses", lambda cpf: 12)]:
    r = vencer(meses)
    print(nome, "| vencido até 01/10/2026:", round(r["vencido"].sum()), "pts de", (r["vencido"] > 0).sum(), "clientes",
          "| débitos que só cabiam com ponto vencido:", round(r["acima_do_valido"].sum()), "pts de",
          (r["acima_do_valido"] > 0).sum(), "clientes")

In [ ]:
# K5b: sem teto, o vencimento segura o saldo? quanto um cliente ganha de verdade em 12 e 18 meses (sem o que a K6 tira; o bug vale 1%)
validos = creditos.assign(valido=creditos["q"] - creditos["falta"])
validos.loc[validos["inflado"], "valido"] = (validos.loc[validos["inflado"], "q"] / 100).round()
validos["quando"] = pd.to_datetime(validos["criado_em"], utc=True)
for meses in (12, 18):
    janela = validos[validos["quando"] >= DATA_BASE - pd.DateOffset(months=meses)]
    ganho = janela.groupby("cpf")["valido"].sum()
    print(meses, "meses | clientes:", len(ganho), "| mediana:", ganho.median(), "| 99% até:", round(ganho.quantile(0.99)),
          "| maior:", ganho.max(), "| créditos do maior:", (janela["cpf"] == ganho.idxmax()).sum())
print("maior crédito válido único:", validos["valido"].max())

### Categoria pela regra × categoria manual (K4c)

Pontos ganhos no ano (créditos, com o bug do app a 1%); a categoria manual é a melhor entre os cadastros do CPF (K1).

In [ ]:
DE_PARA_CATEGORIA = {"bronze": "bronze", "brz": "bronze", "silver": "prata", "prata": "prata", "prata*": "prata",
                     "gold": "ouro", "ouro": "ouro", "ouro ★": "ouro", "vip?": "vip?"}
ORDEM_CATEGORIA = {"(vazio)": 0, "bronze": 1, "vip?": 2, "prata": 3, "ouro": 4}
print("grafias:", clientes["categoria"].value_counts(dropna=False).to_dict())

com_cpf_cat = clientes[clientes["cpf"].notna()].copy()
com_cpf_cat["manual"] = com_cpf_cat["categoria"].str.strip().str.lower().map(DE_PARA_CATEGORIA).fillna("(vazio)")
manual = com_cpf_cat.sort_values("manual", key=lambda s: s.map(ORDEM_CATEGORIA)).groupby("cpf")["manual"].last()

ganhos_ano = (lotes_mov[(lotes_mov["tipo"] == "C") & (lotes_mov["mov"] > 0)]
              .assign(ano=lambda d: d["quando"].dt.year).groupby(["cpf", "ano"])["mov"].sum().unstack(fill_value=0))
categoria = pd.DataFrame({"manual": manual}).join(ganhos_ano, how="left").fillna({2025: 0, 2026: 0})
for ano in [2025, 2026]:
    categoria[f"regra_{ano}"] = pd.cut(categoria[ano], [-1, 1999, 4999, float("inf")], labels=["bronze", "prata", "ouro"])
print("CPFs:", len(categoria), "| Ouro manual com 5.000+ pts em 2025:",
      ((categoria["manual"] == "ouro") & (categoria["regra_2025"] == "ouro")).sum(), "de", (categoria["manual"] == "ouro").sum())
pd.crosstab(categoria["manual"], categoria["regra_2026"], margins=True)

## Anotações para o dicionário de dados

**O que é cada linha**
- `clientes` (2.535): um **cadastro** do Vulto+, não uma pessoa: 131 CPFs têm 2 ou mais cadastros (ex.: a mesma Júlia com 85 pontos num e 246 no outro); 22 e-mails repetidos.
- `antigos` (300): cliente do programa de 2023. 180 também estão na aba Clientes (2 só com o CPF corrigido pela K2). Pela aba "LEIA ISSO", os ATIVOS foram migrados em jan/24.
- `resgates` (596): resgate anotado na planilha, de 01/2023 a 09/2026.
- `pontos` (12.127): movimento no banco, de 01/01/2025 a 01/10/2026. C crédito (11.400), D resgate (565), E estorno (101), AJ ajuste (61).
- `compras` (14.948): compra que pode gerar pontos. Inclui as **vendas das lojas físicas** (origem `pdv`, 7.283), que não aparecem em `vendas.compra_id`: é por aqui que a loja física gera pontos.

**Colunas de `clientes`**

| Coluna | O que é | Problemas vistos |
|---|---|---|
| `cpf` | CPF limpo | 28 vazios (dígito verificador errado; 27 com o CPF certo em outra fonte, ver K2); 131 CPFs repetidos |
| `Pontos` | saldo congelado em 01/01/2025 | formatos `2.203`, `1389 pts`, `2,7k`, `15,0k`; 542 vazios; 26 cadastrados depois de 2025 com pontos |
| `categoria` | Bronze / Prata / Ouro | 14 grafias (`brz`, `Silver`, `prata*`, `Gold`, `ouro ★`, `vip?`); 425 vazias; "colocada na mão" |
| `aceita_msg` | aceita receber mensagem (LGPD) | 7 formas (`x`, `sim`, `1`, `S` / `nao`, `N`, `0`; o `x` é sim, K9b); 827 vazios; 94 "não ligar" na `Obs`, 41 deles marcados como sim |
| `loja_cadastro` | onde se cadastrou | 18 grafias |
| `Data Cadastro` | data do cadastro | 206 só com mês/ano; de 2021 a 2026 |
| `Tel` | telefone | 425 vazios no staging: celular sem DDD na origem, que foi para as pendências (150 com o DDD em outra fonte, K9a) |

**Regras do Vulto+ que o sistema não aplica**
- **Dobro no online:** só os canais `site` e `app` ganharam 2 pontos por real. `web`, `e-commerce`, `ecommerce` e `online` ganharam 1 (ver o cruzamento canal × pontos por real).
- **Ouro +50%:** nunca aplicado.
- **Vencimento** (12 meses; 18 para Ouro): nunca aplicado.
- **Valor do ponto:** o banco usa R$ 0,10; a regra desde mar/2025 é R$ 0,05.
- **Saldo negativo:** 51 CPFs com saldo negativo só pelo banco.

**Bug dos pontos do app**
- **Fato:** entre 08 e 11/06/2026, 15 créditos do app deram pontos em dobro sobre o valor **em centavos**: 1.001.610 pontos. Entre 20 e 23/06/2026, 21 ajustes "correcao bug app v3.4" tiraram 1.138.676 pontos de 20 clientes.
- 11 clientes inflados foram corrigidos; **4 continuam inflados (215.982 pontos)**; 9 clientes levaram a correção sem crédito inflado visível.

**Resgates na planilha × banco**
- Desde 2025: 536 resgates na planilha (484 com CPF, 52 só com nome). Os 239 do "Site" estão todos no banco (o próprio sistema anota na planilha). Nas lojas físicas, 204 de 245 também estão no banco; ~41 aparecem só na planilha.
- **Risco:** descontar os resgates da planilha e do banco conta o mesmo resgate duas vezes.
- **Só na planilha (K7):** com a quantidade corrigida (19 resgates com ponto de milhar: `2.000` tinha virado 2), 443 dos 484 com CPF estão no banco e 41 só na planilha (39.000 pts, R$ 2.075); 39 dos 41 têm uma venda com exatamente aquele desconto, do mesmo CPF, na mesma loja, até 1 dia depois (nos de loja que estão no banco, só 12 de 204). O desconto foi dado e os pontos não saíram.
- **Só com o nome (52, 45.900 pts):** nenhum está no banco. Em 43 a venda com o mesmo desconto dá o CPF, e o nome do cadastro desse CPF confirma (29 com nome completo, 14 só com o primeiro nome); 9 ficam sem cliente (6.600 pts).

**Clientes que só existem no banco**
- 534 CPFs têm movimentos de pontos no banco, mas não estão na aba Clientes. 24 são o CPF certo de um cadastro com dígito errado (K2); os outros parecem cadastrados depois do congelamento.

**Avisos (outras frentes)**
- **→ Vendas:** os 76 pedidos do app repetidos geraram 2 `compra_id` cada (pontos em dobro). Ver `01_vendas`.
- **→ Vendas:** a loja física gera pontos pela tabela `compras` (origem `pdv`), sem ligação direta com `vendas`.

**Problemas de clientes e pontos (lista única; as decisões estão no registro de decisões)**

| # | Problema | Quantos | Afeta |
|---|---|---|---|
| 1 | Mesma pessoa em vários cadastros | 131 CPFs | clientes únicos; soma de pontos |
| 2 | Saldo = planilha congelada + banco; formatos de pontos | 2.535 | passivo de pontos |
| 3 | Dobro no online aplicado só em `site` e `app` | ~2.100 créditos | passivo (a empresa deve mais?) |
| 4 | Ouro +50% e vencimento nunca aplicados | todos | passivo |
| 5 | Valor do ponto: R$ 0,10 × R$ 0,05 | — | passivo em R$ (metade ou o dobro) |
| 6 | Bug do app: 4 clientes ainda inflados (216 mil pontos) | 4 + 9 | passivo |
| 7 | Resgates só na planilha × duplicados com o banco | ~41 + 52 sem CPF | passivo |
| 8 | Clientes antigos também na aba nova | 180 | pontos contados duas vezes? |
| 9 | Categoria colocada na mão, 14 grafias | todos | regra do Ouro |
| 10 | `aceita_msg` vazio ou ambíguo (K9b) | 827 | LGPD |
| 11 | CPFs com pontos no banco que não estão na planilha | 534 | clientes sem cadastro, mas com saldo |

**Desafios da revisão**
- **CPFs com 2+ cadastros:** dos 131, em 60 nenhum cadastro tem pontos e em 71 **os dois (ou mais) têm pontos**; nunca só um. Somar os cadastros extras acrescenta 49.973 pontos. Juntar os cadastros e somar os pontos ou ficar com um só é decisão de negócio (afeta o passivo).
- **CPFs só no banco (534):** nenhum difere em só 1 dígito de um CPF **válido** da planilha, mas 24 diferem em 1 dígito de um dos 28 CPFs com dígito errado: são o CPF certo desses cadastros (K2). Os outros não parecem erros de digitação. 350 começaram em 2025 e 184 em 2026: **parecem clientes que passaram a ganhar pontos sem entrar na planilha** (que ficou congelada como fonte de pontos desde 01/01/2025). Eles têm saldo, mas não têm nome, e-mail nem telefone.

**CPF com dígito verificador errado (K2)**
- **Cadastro (28, 34.447 pts na planilha):** 27 têm exatamente um CPF válido conhecido a 1 dígito, com compras e pontos a partir de 2025 (34.291 pts); 2 desses são outro cadastro da mesma pessoa e 24 estão entre os CPFs só no banco. 1 sem candidato (156 pts).
- **Pontos (120 números, 134 movimentos, 51.672 pts):** o sistema creditava qualquer número, então 56 são telefone do cadastro e 13 têm formato de celular (J1c); 1 é `00000000000` (9 pts); **50 são CPF digitado errado (21.639 pts)**, todos com exatamente um CPF conhecido a 1 dígito (46 com movimentos próprios, 40 de 46 na mesma loja). 27 desses estão com máscara na compra (`cliente_tipo` vazio), por isso a contagem da J1c (só `numero`) ficou baixa.
- **Acaso:** cada número tem de 1 a 3 CPFs válidos a 1 dígito entre ~1 bilhão, e só ~3 mil aparecem nas fontes: bater por acaso é praticamente impossível.

**Mesmo CPF em vários cadastros (K1)**
- **Mesma pessoa:** nos 131 CPFs, mesmo primeiro nome (nome completo igual em 126) e mesmo telefone em todos que têm telefone; e-mail e loja de cadastro sempre diferentes; o novo cadastro vem em média 0,9 ano depois. Nenhum saldo se repete: não são cópias.
- **Pontos fora do 1º cadastro:** 49.973. Os 26 cadastrados depois de 2025 com pontos (K5e) são todos cadastros extras.
- **O que o sistema mostrava:** só o 1º cadastro que o login acha, com o `parse_pontos`. Não mostrava 753.067 dos 3.540.592 pts da planilha: 413.800 de saldos abreviados (viram 0), 286.898 de CPFs que o login não acha (espaço no CPF ou zero à esquerda perdido) e 52.369 de outro cadastro do mesmo CPF.

**Saldo inicial (K5)**
- **Planilha + banco:** o sistema somava a coluna `Pontos` aos movimentos do banco (`calcula_saldo` em `main.py`). Só com o banco, 183 clientes ficariam negativos em algum momento; com o saldo da planilha, 22.
- **Abreviados:** 96 saldos como `2,7k` (erro máximo de ±4.800 pts). **Teto:** 44 clientes com exatamente 15.000 e nenhum acima; nenhuma regra nem o código citam o limite.
- **Duas abas (180):** os saldos nunca são iguais (129 caíram, 47 subiram; 861.952 pts em 2023 × 507.831 em 2025). O sistema só lia a aba `Clientes`. Não há dado de 2024 em nenhuma fonte, e a queda não segue a data da última compra.
- **Só na aba antiga (120):** 451.922 pts, última compra em 2023; ATIVO 30 (105.305 pts), INATIVO 36 (135.689), BLOQUEADO 17 (79.239), sem status 37 (131.689). A `LEIA ISSO` diz que os pontos antigos foram migrados em jan/24 "pra quem estava ATIVO": os 30 ATIVOS deveriam estar na aba Clientes e não estão. 23 voltaram a comprar em 2025–26 (5 ATIVOS), começaram do zero e nenhum resgatou.
- **Cadastrados depois de 2025 com pontos (26, 7.352 pts):** 10 pelo site ou app; todas as compras dos outros ganharam pontos no banco. Origem não explicada.
- **Negativos (22):** 7 voltam em menos de 1 hora, 9 voltam depois e 6 nunca voltam (−2.547 pts). O `/resgate` não checa o saldo.

**CPFs só nos pontos (K8)**
- **510 CPFs** (todos válidos), 988.094 pts, começaram em 2025 (337) e 2026 (173); 1.693 compras (app 876, loja física 787, site 30). 505 aparecem nas vendas, só com o CPF: nenhuma fonte tem nome ou e-mail. 1 termina negativo (−1.485): é o CPF de teste `12345678909`, com créditos de 1–2 pts só de vendas de teste (V8) e 3 resgates de 500 em 3 minutos pelo `/resgate`. Sem ele: 509 CPFs, 989.579 pts.

**Pontos de vendas que não valem e bug do app (K6)**
- **Falta estornar 862.858 pts (658 créditos, 501 clientes, ≈ R$ 43,8 mil):** bug do app sem correção 510.190, cancelada online 122.137, `aguardando` 121.724, cópia (V5) 42.556, estornado/chargeback 33.820, cancelada em loja física 32.431. Ninguém fica negativo.
- **Bug do app:** 33 créditos sobre o valor em centavos, de 08 a 12/06/2026 (1.721.609 pts). A Vulto corrigiu 20 clientes deixando 1% do crédito; 12 ficaram sem correção. O filtro antigo (só o dobro) achava 4.

**Valor do ponto (K3)**
- **O que foi pago:** até fev/2025, os 90 resgates da planilha a R$ 0,10. Depois, os anotados à mão sem CPF a R$ 0,05 (47 de 47) e os do `/resgate` a R$ 0,10 (229 do site e a maioria das lojas): o `valor_ponto` nunca mudou.
- **Saldo por época (FIFO):** 3.538.620 pts ganhos até fev/2025 e 5.291.206 depois, ≈ R$ 618 mil pela K3 (R$ 883 mil tudo a 0,10; R$ 441 mil tudo a 0,05). Gastando os mais novos primeiro: R$ 631 mil.

**Telefone sem DDD e consentimento (K9)**
- **Sem DDD (425):** todos celulares de 9 dígitos. 150 aparecem completos em outra fonte (atendimento, vendas, compras ou outro cadastro), sempre com um DDD só; é a mesma pessoa (primeiro nome igual nos 39 de outro cadastro, 38 com o mesmo CPF, e nos 65 do atendimento com nome), e o acaso daria ~0,01 número. O DDD da cidade da loja acerta só 79 de 93. Restam 275 (130 de loja física, 145 online).
- **Consentimento:** sim/S/1 746, `x` 268, nao/N/0 694, vazio 827. Nos CPFs com mais de um cadastro, `x` aparece com `sim` 19 vezes e nunca com `nao`; `sim` e `nao` nunca aparecem juntos. Com a K9b, por pessoa (2.367): 920 aceitam, 737 não, 710 não informado; 9 têm aceite e recusa e ficam com a recusa.

*(continuar preenchendo ao longo da análise)*

**Vencimento, Ouro e dobro no online (K4)**
- **Dobro:** o sistema só dobra quando o canal é exatamente `site` ou `app`; `E-commerce`, `ecommerce`, `web`, `online` e `APP` creditaram 1 ponto por real. 3.245 créditos de 1.385 clientes sem o dobro (1.918.653 pts); sem os 325 que a K6 estorna, 2.920 créditos de 1.326 clientes, **1.444.367 pts (≈ R$ 76.230,05 pela K3)**.
- **Vencimento:** nunca aplicado (`config.validade_dias = 365`, mas `# TODO expirar pontos`; nenhum movimento de vencimento; a tela não mostra validade). Se fosse aplicado para trás, 4.871.132 pts de 2.091 clientes teriam vencido até 01/10/2026, e 91 clientes resgataram 1.153.518 pts que já teriam vencido (a loja honrava ponto com mais de 12 meses).
- **Categoria:** 14 grafias na coluna `categoria` (preenchida à mão), 425 vazias e 29 `vip?`; a tela mostra o vazio como Bronze. Pela regra (pontos ganhos no ano), em 2026 são 52 Ouro, 297 Prata e 2.018 Bronze; dos 134 Ouro da planilha, só 10 chegam a 5.000 pts em 2026 (11 em 2025). Nenhum crédito tem 1,5 ponto por real: o bônus Ouro nunca foi dado.
